# **Performance optimization**

## **<span style = "color:red">Q1</span>. A Spark notebook that took 20 minutes now takes 2 hours. How do you investigate?**

- **Start with what changed:** data volume, code, a Spark runtime upgrade, or capacity contention.
    - Open the **Monitoring hub** and the **Spark UI**.
    - Look at stage timelines for skew (a few tasks far longer than the rest), spill to disk, excessive shuffle, and many tiny tasks.
- **Then check the Delta table:** file count and file size. A table with tens of thousands of small files is the most common cause of a sudden slowdown.
- **Fix the cause, not the symptom:** compact files, fix the skewed join, or right-size the pool.
- **Confirm** with the same metrics afterward.

## **<span style = "color:red">Q2</span>. How do you handle the small-file problem in a Lakehouse?**

Frequent micro-batches and streaming writes create many small Parquet files. Remedies, in order of preference: 
- enable <span style = "color:blue">**optimize write**</span> at ingestion so files come out larger;
- schedule <span style = "color:blue">**OPTIMIZE**</span> on hot tables (target roughly 128 MB to 1 GB files depending on size);
- run <span style = "color:blue">**VACUUM**</span> to remove unreferenced files after the retention period;
- batch upstream where the business allows.

**Mention the trade-off:** <span style = "color:blue">**OPTIMIZE**</span> itself consumes capacity, so run it on tables that matter, not on everything.

## **<span style = "color:red">Q3</span>. Explain partitioning vs. Z-ordering vs. V-Order.**

- **<span style = "color:blue">Partitioning</span>** physically splits data by a column.
    - Use it only for low-cardinality columns that are almost always in filters (for example, date at day or month level).
    - Over-partitioning creates small files.
- **<span style = "color:blue">Z-ordering</span>** co-locates related values within files to improve data skipping for filters on high-cardinality columns.
    - It is applied during OPTIMIZE.
- **<span style = "color:blue">V-Order</span>** is a Fabric **write-time optimization** of Parquet that speeds up reads in Fabric engines, especially SQL endpoint and Direct Lake.
    - It adds some write cost, so teams often leave it on for Gold tables that are read heavily and consider turning it off for write-heavy Bronze (verify current defaults per workspace and runtime).


**Story:** I used partitioning, Z-ordering, and automated quality checks and improved pipeline reliability by 40%. Be ready to say which columns you chose and why.


## **<span style = "color:red">Q4</span>. How do you fix data skew in a join?**

**Confirm it first:** one or a few tasks dominate the stage.

**Then choose:**
- **<span style = "color:green">Broadcast</span>** the smaller table if it fits;
- Enable adaptive query execution (AQE) skew join handling;
- **<span style = "color:green">Salt</span>** the skewed key;
- **<span style = "color:green">Filter</spaqn>** or <span style = "color:green">**pre-aggregate</span>** before the join;
- <span style = "color:green">**Isolate hot keys**</span> and process them separately.

State the <span style = "color:green">**trade-off**</span>, since salting adds complexity and a final aggregation step.

## **<span style = "color:red">Q5</span>. Direct Lake is slow or the report suddenly gets slower. What do you check?**

**<span style = "color:red">Likely cause</span>:** Fallback to DirectQuery, or cold-cache loading of columns into memory.

**<span style = "color:blue">Check</span>:**
- Table and column size against the capacity's Direct Lake limits,
- Whether the model uses features that force fallback (for example, views or some security configurations on the SQL endpoint),
- File counts and row group sizes in the Delta tables,
- Whether V-Order was applied.

**<span style = "color:green">Fix</span>:** File compaction, V-Order on Gold tables, narrower tables or aggregation tables, and removing unused columns from the semantic model.

## **<span style = "color:red">Q6</span>. How do you tune a Fabric Warehouse query or a Pipeline copy?**

**<span style = "color:blue">Warehouse</span>**:
- check the query plan and statistics,
- avoid SELECT *,
- filter early,
- use appropriate data types and join keys,
- avoid many tiny INSERTs (load in bulk),
- use CTAS for transformations.

**<span style = "color:blue">Pipeline copy</span>**:
- parallel copy and partition options on large sources,
- staging where appropriate,
- pushing filters to the source so you move less data.

## **<span style = "color:red">Q7</span>. How do you design an incremental load?**

- Pick a <span style = "color:brown">**reliable watermark**</span> (modified timestamp or change tracking or CDC).
- Land the delta into Bronze, then <span style = "color:brown">**MERGE**</span> into Silver on the business key.
- <span style = "color:brown">**Handle late-arriving**</span> data with a lookback window, deletes with soft-delete flags or CDC operation columns
- Make the load <span style = "color:brown">**idempotent**</span> so a retry never duplicates rows.
- Record the watermark only after a successful write.

# **Scenario-based production problems** 
    (Format for every scenario: stabilize, diagnose, fix, prevent, communicate.)

## **<span style = "color:red">Scenario 1</span>: The 6 a.m. executive dashboard is empty.**

- **Stabilize:** confirm scope.
    - Is it one report or all?
    - Tell stakeholders you are on it and give an update time.
- **Diagnose:** check.
    - Pipeline run in the Monitoring hub
    - Upstream source availability
    - Capacity (was it throttled overnight?)
    - Semantic model refresh history.
- **Fix:**
    - rerun or backfill the failed stage only
    - refresh the model.
- **Prevent:**
    - freshness checks before the report loads,
    - alerting through Azure Monitor and Log Analytics
    - retry with backoff.
- **Communicate:** short written post-incident review with root cause and follow-up actions.

## **<span style = "color:red">Scenario 2</span>: Capacity is throttled and everyone's queries are slow.**

**Open the <span style = "color:green">Capacity Metrics app</span>:**
- Find which items consumed the most CU, and when.
- Distinguish interactive operations (reports, queries) from background operations (pipelines, Spark, refreshes).
- Fabric smooths usage over time, so a single heavy job can cause delays hours later.

**Short-term:**
- pause or reschedule the heavy job, or scale the SKU temporarily.

**Long-term:**
- move heavy Spark to a separate capacity or to autoscale billing, stagger schedules, and tune the offending job.

<span style = "color:brown">**Always report the cost of the fix, not just the fix.**</span>

## **<span style = "color:red">Scenario 3</span>: A source system changed a column type and the pipeline broke.**

**Issue:** <span style = "color:brown">Schema drift</span>.

**Short-term:**
- fix the mapping, reprocess from Bronze (the reason Bronze keeps raw data).

**Long-term:**
- schema checks at ingestion
- Delta schema evolution used deliberately (not blindly with mergeSchema)
- data contracts with the source team
- quarantine table for rejected records so one bad batch does not block everything.


## **<span style = "color:red">Scenario 4</span>: Duplicate rows appeared in Silver after a retry.**

**Issue:** The load was not <span style = "color:brown">**idempotent**</span>.

**Fix:** 
- Find the duplicate window with a <span style = "color:green">**Delta time-travel**</span> query
- restore or rebuild the affected partitions
- change the write from append to <span style = "color:green">**MERGE**</span> on a deterministic key (or overwrite by partition)
- Add a <span style = "color:green">**uniqueness check**</span> that fails the pipeline.

## **<span style = "color:red">Scenario 5</span>: A Spark job fails with out-of-memory errors.**

**Diagnose:** 
- Check for <span style = "color:red">**skew**</span>, a huge <span style = "color:red">**collect**</span> to the driver, a <span style = "color:red">**broadcast**</span> that is too large, or <span style = "color:red">**wide rows**</span>.

**Fix:**
- Fix the root cause first (<span style = "color:green">**repartition, avoid collect, adjust broadcast threshold**</span>).
- Increase <span style = "color:green">**executor memory**</span> or <span style = "color:green">**node size**</span> only as a last resource, and say what it costs.

## **<span style = "color:red">Scenario 6</span>: Auditors ask who accessed patient data last quarter.**

**Explain layered controls:**
- Workspace roles and item permissions
- Row-level and column-level security
- Dynamic data masking, Encription (Data at rest, Data in transit)
- Sensitivity labels
- Microsoft Purview for lineage and audit
- Activity logs exported to Log Analytics for retention.

Explain what you implemented for <span style = "color:red">**HIPAA**</span> and <span style = "color:red">**SOX**</span> environments

## **<span style = "color:red">Scenario 7</span>: A deployment to production broke a downstream report.**

**Explain release discipline:**
- Git integration
- Deployment pipelines or Azure DevOps with dev, test, and prod workspaces
- Automated tests on a representative dataset
- Parameterized connections.
- Roll back by redeploying the previous version.

**Connect this to:** Implemented CI/CD standards reduced production deployment incidents by 20%.

# **Cost optimization**

**The core idea**
- **Fabric is capacity-based:** you buy Capacity Units (CUs) by SKU (F2 up to F2048), and all workloads share them.
- **Cost control is about:**
    1. buying the right amount
    2. not wasting what you buy, and
    3. isolating workloads so one cannot starve the rest.

<table>
    <tr><th>Lever</th><th>What to say</th></tr>
    <tr><td>Right-size the SKU (Stock Keeping Unit)</td><td>Use the Capacity Metrics app to find real peak and average usage. Many teams over-buy for a peak that lasts an hour.</td></tr>
    <tr><td>Pause and resume</td><td>Pay-as-you-go F SKUs can be paused (dev and test overnight and weekends). Reservations lower the hourly rate for steady production.</td></tr>
    <tr><td>Smoothing and bursting</td><td>Background work is smoothed over 24 hours and Spark can burst well above base CUs, so short peaks do not need a bigger SKU. Know the limit: sustained overuse still causes throttling.</td></tr>
    <tr><td>Autoscale billing for Spark</td><td>An opt-in, pay-as-you-go model where Spark jobs run on dedicated serverless compute billed for execution time, with a maximum CU limit set by the capacity admin. Good for bursty or exploratory Spark; steady recurring jobs may be cheaper on base capacity. It does not burst from or fall back to the capacity. (verify current GA details)</td></tr>
    <tr><td>Spark pool settings</td><td>Use starter pools or small custom pools, set sensible autoscale min and max nodes, enable high-concurrency mode to share sessions, and keep session timeouts short so idle sessions stop billing.</td></tr>
    <tr><td>Efficient data design</td><td>Incremental loads instead of full reloads, compacted files, partition pruning, and removing unused columns from semantic models.</td></tr>
    <tr><td>Storage</td><td>Retention policies, VACUUM, and avoiding unnecessary copies. Use OneLake shortcuts and mirroring instead of duplicating data.</td></tr>
    <tr><td>Workload isolation</td><td>Separate capacities for production reporting, heavy engineering, and development so one runaway job cannot throttle executives.</td></tr>
    <tr><td>Chargeback</td><td>Tag workspaces by team and report CU use by workspace so cost has an owner.</td></tr>
    <tr><td>d</td><td>d</td></tr>
</table>


**Your story to tell (STAR):** 
- At XXX (company), Fabric spend was growing.
- You analyzed compute and storage usage
- tuned the workloads
- cut annual infrastructure spend by $120K without breaking SLAs.

**Prepare three specifics before the interview:**
- what the biggest cost driver was
- what you changed
- how you proved SLAs held

**Likely follow-up:** 
- How do you decide between scaling the SKU and optimizing the job?
- **Answer:**
    - optimize first if one workload dominates CU consumption;
    - scale or isolate if usage is broadly healthy but peaks overlap.
    - Always compare engineering effort cost with the monthly capacity cost.

# **Why Fabric stands out, and where it fits?**

**Summary:**
- Fabric puts ingestion, engineering, warehousing, real-time analytics, data science, and BI on one <span style = "color:green">**SaaS platform**</span> with one storage layer (<span style = "color:green">**OneLake**</span>, open Delta Parquet), <span style = "color:green">**one security and governance mode**</span>**, and <span style = "color:green">**one capacity bill**</span>. That removes most of the integration work teams do between separate Azure services.


**What actually differentiates it:**
- OneLake and open formats. One logical lake per tenant, Delta Parquet underneath, and shortcuts that reference data in ADLS, S3, or other sources without copying.
- Direct Lake. Power BI reads Delta files directly with import-like speed and without a scheduled import refresh. A big advantage for large models.
- Unified governance. Purview integration, sensitivity labels that follow data, and consistent security across engines.
- Multiple engines on one copy of data. Spark, SQL, KQL, and Power BI all read the same tables.
- Real-Time Intelligence. Eventstreams and Eventhouse (KQL) for streaming and time-series scenarios in the same platform.
- Low operational overhead. No clusters to patch, simple capacity model, built-in monitoring.
- Mirroring. Near-real-time replication from operational databases into OneLake (verify the source list).

**Trade-offs:**
- **Contention:** Capacity sharing means noisy-neighbor risk unless you isolate workloads.
- **Maturity:** It is younger than Databricks or Snowflake; some advanced features (fine-grained Spark tuning, mature ecosystem tooling, some governance depth) are still catching up.
- **Microsoft-centric:** Strongest when the organization already uses Microsoft 365, Power BI, and Azure. Less compelling for multi-cloud shops.
- **Chargeback:** Cost can be hard to attribute without deliberate workspace design.

**Fabric vs. alternatives:**
- **<span style = "color:green">Synapse</span>:** Fabric is the successor direction: simpler, unified, and SaaS rather than assembling services.
- **<span style = "color:green">Databricks</span>:** Databricks offers deeper Spark and ML control and multi-cloud reach; Fabric wins on integrated BI, simplicity, and Microsoft alignment. Many companies use both.
- **<span style = "color:green">Snowflake</span>:** Snowflake is strong on warehousing ease and cross-cloud sharing; Fabric is stronger when the stack is Power BI and Azure centered.


**Use cases:**
- **Healthcare analytics:** medallion pipelines over clinical and operational data with PHI protections.
- **Legacy modernization:** on-premises SQL Server data warehouse to Fabric Warehouse and OneLake, with Power BI semantic models.
- **Operational reporting:** With X% fresher data and Y% less manual processing. 
- **Other strong fits to mention:**
    - Real-time monitoring with Eventstreams and Eventhouse
    - <span style = "color:green">**Self-service analytics**</span> through Direct Lake
    - Data sharing across business units with <span style = "color:green">**shortcuts**</span>
    - <span style = "color:green">**AI-ready Gold datasets**</span> for data science teams.